<div style="text-align: right"> <font color='Gray'> Laboratorio IV - 2026 </div>
<div style="text-align: right"> <font color='Gray'> Guía Choroplet e importación </div>
<div style="text-align: right"> <font color='Gray'> Sebastián Pulgares </div>


***

# Imports

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.express as px
import matplotlib.pyplot as plt
# import plotly.graph_objects as go 

# Data Censo

## La data del censo se encuentra en: https://censo2024.ine.gob.cl/resultados/
### En particular, para este ejemplo se usa: Base de microdatos - Personas Censo 2024 (csv), hay que descargarla y extraerla

Dado que los archivos son muy grandes y es posible que nuestro computador no pueda mantenerlos por completo en memoria para trabajar, veremos un proceso simple de identificación y filtrado de información relevante

In [ ]:
df = pd.read_csv("../data/raw/personas_censo2024.csv", nrows=5, sep=";") # por lo general, los df en español usan ; como separador

In [ ]:
df.columns

In [ ]:
df.info()

Con esto, identificamos las columas de interés, en este caso, contaremos el numero de personas por comuna

In [ ]:
df = pd.read_csv("../data/raw/personas_censo2024.csv", sep=";",usecols=["id_persona", "comuna"])

In [ ]:
df.info()

In [ ]:
df.head()

# Geopandas
Esta biblioteca es una extensión de Pandas para trabajar con data geoespacial, su documentación está en: https://geopandas.org/en/stable/docs.html.

### Los mapas vectoriales oficiales de Chile se encuentran en: https://www.bcn.cl/siit/mapas_vectoriales

In [ ]:
geodata = gpd.read_file("../data/raw/Comunas/comunas.shp")

In [ ]:
geodata.info()

# Gráficos

A modo de ejemplo, se van a graficar las comunas de una región, el color variará según la población

In [ ]:
personas_por_comuna = df.value_counts("comuna")

Mezcalamos las 2 fuentes de datos

In [ ]:
personas_por_comuna = geodata.merge(personas_por_comuna, left_on="cod_comuna", right_on="comuna")

In [ ]:
personas_por_comuna.head(1)

In [ ]:
random_selection = np.random.choice(personas_por_comuna["Region"])
data_region = personas_por_comuna[personas_por_comuna['Region'] == random_selection]

if random_selection == "Región de Valparaíso":
    islas = ["Isla de Pascua", "Juan Fernández"]
    continental = data_region[~data_region["Comuna"].isin(islas)]
    data_region = continental

## Matplotlib

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
data_region.plot(
    ax=ax,
    column="count",
    cmap='Purples',
    legend=True,
    edgecolor='black'
)
ax.set_xticks([])
ax.set_yticks([])
plt.title(random_selection)
plt.show()

##  Plotly

In [ ]:
data_region = data_region.to_crs(epsg=4326)     #Para usar plotly es necesario pasar a formato estandar de referenciación
data_region = data_region.set_index("Comuna")

fig = px.choropleth(
    data_region,
    geojson=data_region.geometry,
    locations=data_region.index,
    projection="mercator",
    color="count",
    title= random_selection,
    hover_data={"count": True},
    color_continuous_scale="purples" # https://masamasace.github.io/plotly_color/
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_traces(hovertemplate="%{location}\n <br>Población: %{customdata[0]}")
fig.update_layout(width=900, height=700)

fig.show()